# Upstream NISAR product discovery handoff

This optional example keeps product acquisition outside SnowIn:

```text
nisar-pytools → search / download / validate → local GUNW path → SnowIn
```

Install `nisar-pytools` (included in SnowIn's `nisar` extra) and the `notebooks` dependency group. ASF search/download may require locally configured Earthdata credentials. The next notebook, `02_real_nisar_gunw_workflow.ipynb`, demonstrates the SnowIn processing steps.

In [ ]:
import os
from pathlib import Path

from nisar_pytools import download_urls, find_nisar

aoi = [
    float(value)
    for value in input("AOI west,south,east,north: ").split(",")
]
if len(aoi) != 4:
    raise ValueError("Enter four AOI bounds: west,south,east,north")
start_date = input("Start date (YYYY-MM-DD): ").strip()
end_date = input("End date (YYYY-MM-DD): ").strip()
product_type = "GUNW"


## 1. Search upstream

`find_nisar` is called directly from `nisar-pytools`. The SnowIn package does not own this search or choose acquisition filters for your study.

In [ ]:
urls = find_nisar(
    aoi=aoi,
    start_date=start_date,
    end_date=end_date,
    product_type=product_type,
    max_results=20,
)
print(f"Found {len(urls)} GUNW products")
for index, url in enumerate(urls):
    print(index, url.rsplit("/", 1)[-1])

## 2. Select, download, and validate one product

Review the results, then choose one product. `download_urls` performs the upstream download and validation; the GUNW is saved in a caller-owned cache directory.

In [ ]:
if not urls:
    raise RuntimeError("No GUNW products matched the search.")
selected_index = int(input("Index of the GUNW to download: "))
if selected_index not in range(len(urls)):
    raise IndexError("Select one of the listed product indices.")
output_dir = Path.home() / ".cache" / "snowin" / "nisar-products"
files = download_urls(
    [urls[selected_index]], output_dir, max_workers=1, validate=True
)
if not files:
    raise RuntimeError("No product was downloaded and validated.")
gunw_path = Path(files[0]).expanduser().resolve()
if not gunw_path.is_file():
    raise FileNotFoundError(gunw_path)
os.environ["SNOWIN_GUNW"] = str(gunw_path)
print("Validated local GUNW:", gunw_path)
print("Cache directory:", output_dir)
print(f'SNOWIN_GUNW="{gunw_path}"')
print(
    "For terrain-local snow analysis, also set SNOWIN_NISAR_DEM to the "
    "matching NISAR-modified Copernicus DEM."
)
